In [ ]:
# IOGP Life-Saving Rule alert editor — single-cell build (Rust 1.98 + CMake), tests, MCC, Lean proofs, editor + API, clingo review, inference runtime.
# Re-run this cell to re-attach after a disconnect. Optional settings (set before running):
#   os.environ['FORCE']='1' rerun a finished job · ['RUN_LEAN']='0' skip proofs · ['KEEP_SERVER']='1' keep the editor on :9191
import os, subprocess, sys
BUNDLE = '''\
...
'''
SCRIPT = r'''# IOGP Life-Saving Rule alert editor: controlled English (ACE, parsed by APE via ape-rs on Scryer
# Prolog) -> time-indexed FOL + detector configuration. Full reproduction in ONE cell:
#  1. builds ape-rs, acewiki-rs (chart parser) and iogp-editor; runs the unit/integration tests
#  2. compiles the 9 IOGP LSR templates (readback, FOL, JSON config)
#  3. MCC: acceptance set; parity against yappu-fsm's own EventEngine; synthetic site at 0/10/25/40% missed detections
#  4. language: named zones, camera scoping, "unless <permit>" exceptions
#  5. replay of a recorded site (raw boxes per camera + zone polygons + permit timeline) with labelled
#     incidents: recall, false alarms/h, latency, frame MCC  (synthetic sample; point it at a real export)
#  6. checks the Lean 4 proofs (formal/IogpAlerts.lean)
#  7. starts the editor: approval workflow (author submits, a second person approves, four-eyes enforced),
#     hash-chained audit log + tamper test, exports, OpenAI-compatible /v1/chat/completions, /v1/simulate,
#     and the corpus report of what users wrote
#  8. inference runtime crate (no APE/Prolog): fast engine == reference on 72k frames, speed, C ABI demo
#  9. clingo (clingox) offline review: example scenes, dead rules, implications, coverage; rules.lp export
# Runs as a Python cell: Colab's %%shell passes the cell as one argument, and the embedded sources exceed the
# kernel's 128 KB per-argument limit, so the sources and script are written to files and run from there.
# ~20-30 min on Colab CPU (mostly compiling Scryer). Disconnect-safe: the work runs as a detached job
# logging to $W/run.log; if the connection drops, RUN THIS CELL AGAIN to re-attach or resume.
# FORCE=1 reruns a completed job. RUN_LEAN=0 skips the proofs. KEEP_SERVER=1 leaves the editor running
# on port 9191 (open it with google.colab.output.serve_kernel_port_as_window(9191)).
set -uo pipefail
[ -d /content ] && W=${W:-/content/iogp-work} || W=${W:-$PWD/iogp-work}
mkdir -p "$W"; [ -s /tmp/iogp_bundle.b64 ] && cp /tmp/iogp_bundle.b64 "$W/bundle.b64"; LOG="$W/run.log"; JOB="$W/job.sh"; PIDF="$W/job.pid"
alive() { [ -s "$PIDF" ] && kill -0 "$(cat "$PIDF")" 2>/dev/null; }
if alive; then
  echo ">> re-attaching to the running job (pid $(cat "$PIDF")); log so far:"
else
  if [ "$(cat "$W/job.exit" 2>/dev/null)" = 0 ] && [ "${FORCE:-0}" != 1 ]; then
    echo ">> job already completed (FORCE=1 to rerun) - log:"; cat "$LOG"; exit 0
  fi
  if [ -s "$LOG" ]; then
    echo ">> previous run stopped: $( [ -f "$W/job.exit" ] && echo "exit code $(cat "$W/job.exit")" || echo "killed (runtime restart or out of memory)" ), last step: $(cat "$W/status" 2>/dev/null)"
    tail -n 12 "$LOG"; echo ">> resuming after the last completed step"; mv "$LOG" "$LOG.prev"
  fi
  [ "${FORCE:-0}" = 1 ] && rm -rf "$W/.done"
  cat > "$JOB" <<'JOBEOF'
#!/bin/bash
set -euo pipefail
W=${W:?}; cd "$W"
RUN_LEAN=${RUN_LEAN:-1}; KEEP_SERVER=${KEEP_SERVER:-0}
export LANG=C.UTF-8 LC_ALL=C.UTF-8 NO_PROXY='*' DEBIAN_FRONTEND=noninteractive
mkdir -p "$W/.done"
done_() { if [ -f "$W/.done/$1" ]; then echo "  (already done: $1 - skipped)"; return 0; fi; return 1; }
mark() { touch "$W/.done/$1"; }
step() { echo; echo "=== $* ===   [$(date +%H:%M:%S), RAM available $(awk '/MemAvailable/{printf "%d MB", $2/1024}' /proc/meminfo)]"; echo "$*" > "$W/status"; }
trap 'echo "!! FAILED at line $LINENO in step: $(cat "$W/status" 2>/dev/null)"' ERR

step toolchains
if ! done_ toolchains; then
apt-get -qq update >/dev/null && apt-get -qq install -y zstd xz-utils git build-essential pkg-config curl cmake python3-numpy >/dev/null || true
command -v rustup >/dev/null || [ -x "$HOME/.cargo/bin/rustup" ] || curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal --default-toolchain none >/dev/null
mark toolchains; fi
export PATH="$HOME/.cargo/bin:$PATH" CARGO_TARGET_DIR="$W/target"
# the whole stack is pinned to Rust 1.98 (rust-toolchain.toml); clingo is built with CMake
IGNORE_RV=""
rustup toolchain install 1.98 --profile minimal >/dev/null 2>&1 || true
if ! rustup run 1.98 rustc --version >/dev/null 2>&1; then
  if [ "${IOGP_ALLOW_OLD_RUST:-0}" = 1 ]; then
    export RUSTUP_TOOLCHAIN=stable IGNORE_RV="--ignore-rust-version"
    echo "  WARNING: Rust 1.98 could not be installed here; IOGP_ALLOW_OLD_RUST=1 -> building with $(rustc --version) and --ignore-rust-version"
  else
    echo "!! Rust 1.98 could not be installed (rustup toolchain install 1.98 failed). Check network access to static.rust-lang.org."; exit 1
  fi
fi
command -v cmake >/dev/null || { echo "!! CMake is required (clingo is built from source): apt-get install cmake"; exit 1; }

step "sources (ape-rs, acewiki-rs, iogp-editor, yappu-fsm reference)"
if ! done_ bundle; then
base64 -d "$W/bundle.b64" | tar -xJf - -C "$W"
mark bundle; fi
ls -d ape-rs acewiki-rs iogp-editor yappu-fsm
P="$W/ape-rs/prolog-scryer"; export APE_PROLOG_DIR="$P"

step "cargo build + tests (Scryer is fetched from dhaneshkk/scryer-prolog@fix/ape-rs-findings, pinned in Cargo.lock)"
AVAIL_GB=$(( $(awk '/MemAvailable/{print $2}' /proc/meminfo) / 1048576 ))
JOBS=$(( AVAIL_GB / 3 )); [ "$JOBS" -lt 1 ] && JOBS=1; [ "$JOBS" -gt "$(nproc)" ] && JOBS=$(nproc)
export CARGO_BUILD_JOBS=$JOBS CARGO_TERM_PROGRESS_WHEN=never RUSTFLAGS="-A warnings"
echo "  RAM available ${AVAIL_GB} GB -> cargo jobs $JOBS"
cd "$W/iogp-editor"
RUSTUP_TOOLCHAIN=${RUSTUP_TOOLCHAIN:-1.98} ./scripts/check-toolchain.sh | sed 's/^/  /' || [ -n "$IGNORE_RV" ]
time cargo build --release -q --bins $IGNORE_RV
cargo test --release -q $IGNORE_RV > "$W/test.log" 2>&1 || { tail -n 40 "$W/test.log"; echo "tests FAILED"; exit 1; }
grep -E "^test result" "$W/test.log" | grep -v " 0 passed" | sed 's/^/  /' 
B="$CARGO_TARGET_DIR/release"

step "the 9 IOGP Life-Saving Rule templates: readback"
python3 - > /tmp/templates.ace <<'PY'
import json
for t in json.load(open("data/vg360_catalog.json"))["templates"]: print(f"LSR{t['lsr']}: {t['text']}")
PY
"$B/iogp" compile --file /tmp/templates.ace --format readback
echo; echo "--- FOL (TPTP) of LSR4 and LSR7"
"$B/iogp" compile --format tptp "$(sed -n 4p /tmp/templates.ace | cut -d' ' -f2-)" "$(sed -n 7p /tmp/templates.ace | cut -d' ' -f2-)" | grep -v '^%' | sed '/^$/d'

step "MCC 1: acceptance set (valid / invalid sentences)"
python3 - "$B/iogp" <<'PY'
import subprocess, sys, math, re
items = [(l[0] == "+", *l[1:].split(": ", 1)) for l in open("data/acceptance.ace") if l[:1] in "+-"]
open("/tmp/acc.ace", "w").write("".join(f"{i}: {s}" for _, i, s in items))
r = subprocess.run([sys.argv[1], "compile", "--file", "/tmp/acc.ace", "--format", "readback"], capture_output=True, text=True)
why, cur = {}, None
for l in r.stderr.splitlines():
    m = re.match(r"✗ ([^:]+): ", l)
    if m: cur = m.group(1); why[cur] = []
    elif cur and l.startswith("    "): why[cur].append(l.strip())
tp=tn=fp=fn=0
for want, i, s in items:
    got = i not in why
    tp+=want and got; tn+=(not want) and (not got); fp+=(not want) and got; fn+=want and (not got)
    if want != got: print("  MISMATCH", i, s.strip())
    if not want: print(f"  rejected {i:6s} ->", (why.get(i) or ["?"])[0][:150])
den = math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)) or 1
print(f"  acceptance: tp={tp} tn={tn} fp={fp} fn={fn}  MCC={(tp*tn-fp*fn)/den:.4f}")
PY

step "MCC 2: parity with yappu-fsm's own vision EventEngine (yappu-compatible mode)"
python3 tools/yappu_parity.py "$W/yappu-fsm" /tmp/parity.jsonl
"$B/iogp_eval" parity /tmp/parity.jsonl | tail -n 6

step "MCC 3: synthetic site, 9 LSR rules, detections dropped at random (engine vs yappu timers vs per-frame)"
for m in 0.00 0.10 0.25 0.40; do echo "--- missed detections $m"; "$B/iogp_eval" synthetic --miss $m | tail -n 12; done

step "language: named zones, camera scoping, permit exceptions"
cat > /tmp/lang.ace <<'ACE'
NAMED: If a worker is in Rig-Floor-East and does not wear a harness for at least 3 seconds then VisionGuard raises a critical working-at-height-alert for the worker.
CAMERA: If a worker is in a height-zone of CAM-03 and does not wear a harness for at least 3 seconds then VisionGuard raises a critical working-at-height-alert for the worker.
UNLESS: If a worker is in Moonpool for at least 5 seconds then VisionGuard raises a high work-authorisation-alert for the worker unless Moonpool has a permit or the worker has a permit.
BAD1: If a worker is in a height-zone then VisionGuard raises a critical working-at-height-alert for the worker unless the worker wears a harness.
BAD2: If a worker is in a restricted-area then VisionGuard raises a high work-authorisation-alert for the worker unless the hot-work-area has a permit.
ACE
"$B/iogp" compile --file /tmp/lang.ace --format readback || true
"$B/iogp" compile --format tptp "$(sed -n 3p /tmp/lang.ace | cut -d' ' -f2-)" | grep -v '^%' | sed '/^$/d'

step "replay a recorded site with labelled incidents (SYNTHETIC sample in the raw VisionGuard360-style format)"
python3 tools/sample_vg360_log.py /tmp/site
head -c 400 /tmp/site/detections.jsonl; echo " ..."
"$B/iogp_eval" --report /tmp/site/report.json replay --site /tmp/site/site.json --detections /tmp/site/detections.jsonl --rules /tmp/site/rules.ace --labels /tmp/site/labels.jsonl
echo "  -> with a real export: iogp_eval replay --site site.json --detections export.jsonl --store iogp_rules.json --labels officer_labels.jsonl"

step "offline rule review with clingo (clingox): example scenes, dead rules, implications, coverage"
cat > /tmp/review.ace <<'ACE'
WAH: If a worker is in a height-zone and does not wear a harness for at least 3 seconds then VisionGuard raises a critical working-at-height-alert for the worker.
WAHE: If a worker is in Rig-Floor-East and does not wear a harness for at least 3 seconds then VisionGuard raises a critical working-at-height-alert for the worker.
MOON: If a worker is in Moonpool for at least 5 seconds then VisionGuard raises a high work-authorisation-alert for the worker unless Moonpool has a permit.
CROWD: If at least 3 workers are in a lifting-zone for at least 5 seconds then VisionGuard raises a high mechanical-lifting-alert.
CROWD2: If at least 2 workers are in a lifting-zone for at least 5 seconds then VisionGuard raises a medium mechanical-lifting-alert.
NEVER: If a worker is in a height-zone and is not in the height-zone then VisionGuard raises a critical working-at-height-alert for the worker.
ACE
"$B/iogp" review --file /tmp/review.ace
(cd "$W/iogp-editor" && cargo test --release -q $IGNORE_RV --test asp_differential -- --nocapture 2>&1 | grep identical | sed 's/^/  /') || true
echo "--- rules.lp (first rule of the 9 templates)"; "$B/iogp" lp --file /tmp/templates.ace | sed -n '1,20p'

if [ "$RUN_LEAN" = 1 ]; then
  step "Lean 4 proofs"
  LEAN=/opt/lean/lean-4.25.0-linux/bin/lean
  [ -x "$LEAN" ] || { curl -sSfL https://github.com/leanprover/lean4/releases/download/v4.25.0/lean-4.25.0-linux.tar.zst -o /tmp/lean.tar.zst \
                      && mkdir -p /opt/lean && tar --use-compress-program=unzstd -xf /tmp/lean.tar.zst -C /opt/lean; }
  "$LEAN" formal/IogpAlerts.lean && echo "IogpAlerts.lean: $(grep -c '^theorem' formal/IogpAlerts.lean) theorems checked, $(grep -c 'sorry' formal/IogpAlerts.lean) sorry"
fi

step "editor: approval workflow, audit log, exports, OpenAI-compatible endpoint, simulate"
pkill -f "iogp --prolog-dir" || true; rm -f "$W"/iogp_rules*.json* "$W"/users.json
"$B/iogp" user add omar --role author --password pw-omar --file "$W/users.json" >/dev/null
"$B/iogp" user add dana --role approver --password pw-dana --file "$W/users.json" >/dev/null
nohup "$B/iogp" --prolog-dir "$P" serve --addr 0.0.0.0:9191 --rules "$W/iogp_rules.json" --users "$W/users.json" > "$W/serve.log" 2>&1 &
SRV=$!
for _ in $(seq 120); do curl -sf 127.0.0.1:9191/healthz >/dev/null && break; sleep 1; done
U=http://127.0.0.1:9191
curl -s -c /tmp/omar.jar -o /dev/null -X POST $U/login -d name=omar -d password=pw-omar
curl -s -c /tmp/dana.jar -o /dev/null -X POST $U/login -d name=dana -d password=pw-dana
echo "  wrong password ->" $(curl -s -o /dev/null -w '%{redirect_url}' -X POST $U/login -d name=dana -d password=x)
sug() { curl -s $U/suggest -H 'Content-Type: application/json' -d "$(python3 -c 'import json,sys;print(json.dumps({"text":sys.argv[1]}))' "$1")" | python3 -c "
import json,sys; s=json.load(sys.stdin)
print('  %-58s -> %s%s%s' % (repr(sys.argv[1][-56:]), ', '.join(o['w'] for o in s['options'][:6]), ('   ghost: '+repr(s['ghost'])) if s['ghost'] else '', ('   ERROR: '+s['error']) if s.get('error') else ''))" "$1"; }
echo "--- auto-suggestion (/suggest): next words for the text so far, typed-prefix filter, ghost completion, spelling fixes"
sug "If a wor"
sug "If a worker is in a hei"
sug "If a worker is in a height-zone and does no"
sug "If a worker is in a height-zone and does not wear a "
sug "If a worker is in a height-zone and does not wear a harness for at least 3 seconds then"
sug "If a worker is in a height-zone and does not wear a harness for at least 3 seconds then VisionGuard raises a crtical"
sug "If a worker wears a permit"
red() { grep -i '^hx-redirect' | sed 's/.*msg=//' | python3 -c "import sys,urllib.parse;print('  ->', urllib.parse.unquote(sys.stdin.read().strip()))"; }
while IFS= read -r l; do id=${l%%: *}; txt=${l#*: }; curl -s -b /tmp/omar.jar -o /dev/null -X POST $U/rules --data-urlencode "id=$id" --data-urlencode "text=$txt" -d op=submit; done < /tmp/templates.ace
echo "  9 templates submitted by omar; live rules before approval:" $(curl -s $U/export/rules.ace | wc -l)
echo "  omar approves his own LSR9:"; curl -s -b /tmp/omar.jar -D - -o /dev/null -X POST $U/rules/LSR9/1/approve | red
for k in 1 2 3 4 5 6 7 8 9; do curl -s -b /tmp/dana.jar -o /dev/null -X POST $U/rules/LSR$k/1/approve -H "HX-Prompt: reviewed against IOGP 459 rule $k"; done
echo "  dana approved all 9; live rules:" $(curl -s $U/export/rules.ace | wc -l)
curl -s -b /tmp/omar.jar -o /dev/null -X POST $U/rules --data-urlencode "id=LSR9" --data-urlencode "text=$(sed -n 2p /tmp/lang.ace | cut -d' ' -f2-)" -d op=submit --data-urlencode "intent=only the east rig floor camera"
echo "  omar submits LSR9 v2 (camera-scoped); live text unchanged until approved:" $(curl -s $U/export/rules.ace | grep '^LSR9' | cut -c1-60)
echo "  dana rejects it:"; curl -s -b /tmp/dana.jar -D - -o /dev/null -X POST $U/rules/LSR9/2/reject -H 'HX-Prompt: keep the rule site-wide' | red
for u in / /review /audit /analysis /rules/LSR9 /catalog /grammar.json /export/config.json /export/rules.p /export/rules.lp /export/rules.ace /export/audit.jsonl /v1/models; do printf "  GET %-20s %s\n" "$u" "$(curl -s -b /tmp/dana.jar -o /dev/null -w '%{http_code} %{size_download}B' $U$u)"; done
curl -s $U/export/config.json > "$W/config.json"
curl -s $U/export/config.json | python3 -c "import json,sys;d=json.load(sys.stdin);print('  config.json:',d['schema'],'| revision',d['revision'][:16],'|',len(d['rules']),'approved rules | LSR9 approved by',[r for r in d['rules'] if r['id']=='LSR9'][0]['version']['approved_by'])"
"$B/iogp" audit verify "$W/iogp_rules.audit.jsonl"
cp "$W/iogp_rules.audit.jsonl" /tmp/tampered.jsonl; sed -i '3s/"user":"omar"/"user":"mallory"/' /tmp/tampered.jsonl
echo "  a copy with entry 3's author changed:"; "$B/iogp" audit verify /tmp/tampered.jsonl && { echo "TAMPER NOT DETECTED"; exit 1; } || true
sed -i '5d' /tmp/tampered.jsonl; cp "$W/iogp_rules.audit.jsonl" /tmp/tampered2.jsonl; sed -i '5d' /tmp/tampered2.jsonl
echo "  a copy with entry 5 deleted:"; "$B/iogp" audit verify /tmp/tampered2.jsonl && { echo "TAMPER NOT DETECTED"; exit 1; } || true
chat() { curl -s $U/v1/chat/completions -H 'Content-Type: application/json' -d "$(python3 -c 'import json,sys;print(json.dumps({"model":sys.argv[1],"messages":[{"role":"user","content":sys.argv[2]}]}))' "$1" "$2")" | python3 -c "import json,sys;print(json.load(sys.stdin)['choices'][0]['message']['content'])"; }
echo "--- iogp-rule-explain"
chat iogp-rule-explain $'A1: If a worker is on a ladder and does not wear a harness for at least 3 seconds then VisionGuard raises a critical working-at-height-alert for the worker.\nA2: If a worker is in a hot-work-area and does not wear a helmt for at least 5 seconds then VisionGuard raises a medium hot-work-alert for the worker.'
echo "--- iogp-rule-check"
chat iogp-rule-check $'If a worker holds a phone for more than 10 seconds then VisionGuard raises a low driving-alert for the worker.\nIf a worker does not wear a harness then VisionGuard raises a critical working-at-height-alert for the worker.'
echo "--- iogp-rule-compiler (FOL + config for the detection framework)"
chat iogp-rule-compiler 'H1: If at least 2 workers are in a confined-space for at least 30 seconds then VisionGuard raises a high confined-space-alert.' | python3 -c "import json,sys;d=json.load(sys.stdin);r=d['rules'][0];print(r['fol']);print('  conditions:',json.dumps(r['conditions']));print('  requires:',json.dumps(r['requires']))"
echo "--- /v1/simulate: harness seen until 0.9 s, worker stays in the height-zone (grace 0.7 s, LSR9 needs 3 s)"
python3 - <<'PY' > /tmp/sim.json
import json
fr=[]
for i in range(70):
    t=round(i/10,1); objs=[{"id":"p1","class":"person"},{"id":"z1","zone":"height-zone"}]; facts=[["in","p1","z1"]]
    if t<1: objs.append({"id":"h1","class":"harness"}); facts.append(["wear","p1","h1"])
    fr.append({"t":t,"objects":objs,"facts":facts})
print(json.dumps({"frames":fr}))
PY
curl -s $U/v1/simulate -H 'Content-Type: application/json' -d @/tmp/sim.json | python3 -c "import json,sys;[print('  ',e['kind'],e['rule'],'t=%.1f'%e['t'],e['subject'],'|',' / '.join(e['proof'][1:])) for e in json.load(sys.stdin)['events']]"
echo "   (expected: raised at t=4.7 - harness still believed at 1.6 s (gap = grace), absent from 1.7 s, then 3 s)"
echo "--- corpus: what users wrote (compile attempts + intents)"
chat iogp-rule-check 'If a worker is on a scaffold and does not wear a harness for at least 3 seconds then VisionGuard raises a critical working-at-height-alert for the worker.' >/dev/null
"$B/iogp_eval" corpus "$W/iogp_rules.attempts.jsonl" --store "$W/iogp_rules.json" | head -n 20
if [ "$KEEP_SERVER" = 1 ]; then echo "editor left running on :9191 (pid $SRV)"; else kill $SRV || true; fi
step "inference runtime: iogp-runtime (no APE/Prolog) — fast engine vs reference, speed, C ABI on the approved config.json"
cd "$W/iogp-editor/iogp-runtime"
cargo test --release -q $IGNORE_RV > "$W/rt-test.log" 2>&1 || { tail -n 40 "$W/rt-test.log"; echo "runtime tests FAILED"; exit 1; }
grep -E "^test result" "$W/rt-test.log" | grep -v " 0 passed" | sed 's/^/  /'
cargo test --release -q $IGNORE_RV --test differential -- --nocapture 2>&1 | grep identical | sed 's/^/  fast vs reference: /'
cargo run --release -q $IGNORE_RV --example bench
cargo build --release -q $IGNORE_RV   # cargo test does not produce the .a/.so artefacts
cc -O2 examples/c/demo.c -Iinclude "$CARGO_TARGET_DIR/release/libiogp_runtime.a" -lpthread -ldl -lm -o "$W/iogp_demo"
echo "  C demo on the editor's exported config.json (approved rules only):"; "$W/iogp_demo" "$W/config.json" | sed 's/^/    /'
ls -la "$CARGO_TARGET_DIR/release/libiogp_runtime.so" "$CARGO_TARGET_DIR/release/libiogp_runtime.a" | awk '{printf "  %s %.1f MB\n", $9, $5/1048576}'
cd "$W/iogp-editor"
echo; echo "DONE. Start the editor with: $B/iogp --prolog-dir $P serve --addr 0.0.0.0:9191 --rules $W/iogp_rules.json"
JOBEOF
  rm -f "$W/job.exit"
  W="$W" RUN_LEAN="${RUN_LEAN:-1}" KEEP_SERVER="${KEEP_SERVER:-0}" setsid nohup bash -c 'bash "$0"; echo $? > "$1/job.exit"' "$JOB" "$W" > "$LOG" 2>&1 < /dev/null &
  echo $! > "$PIDF"
  echo ">> started detached job (pid $!), log: $LOG"
fi
n=0; t0=$(date +%s); quiet=0
while :; do
  cur=$(wc -l < "$LOG" 2>/dev/null || echo 0)
  if [ "$cur" -gt "$n" ]; then sed -n "$((n+1)),${cur}p" "$LOG"; n=$cur; quiet=0; fi
  alive || break
  sleep 10; quiet=$((quiet+10))
  if [ "$quiet" -ge 60 ]; then
    printf '   ... %s | %d min | step: %s | RAM avail %s MB\n' "$(date +%H:%M:%S)" $(( ($(date +%s)-t0)/60 )) "$(cat "$W/status" 2>/dev/null)" "$(awk '/MemAvailable/{printf "%d", $2/1024}' /proc/meminfo)"
    quiet=0
  fi
done
sleep 1; cur=$(wc -l < "$LOG"); [ "$cur" -gt "$n" ] && sed -n "$((n+1)),${cur}p" "$LOG"
rc=$(cat "$W/job.exit" 2>/dev/null || echo "?"); echo; echo ">> job finished with exit code $rc"; [ "$rc" = 0 ]
'''
open('/tmp/iogp_bundle.b64', 'w').write(BUNDLE)
open('/tmp/iogp_cell.sh', 'w').write(SCRIPT)
p = subprocess.Popen(['bash', '/tmp/iogp_cell.sh'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
for line in p.stdout:
    print(line, end='', flush=True)
rc = p.wait()
if rc != 0:
    raise SystemExit(f'iogp job failed with exit code {rc} - see the log above')
